In [ ]:

import os
import cv2
import timm
import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm



SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)



BASE_DIR = Path("./")

CSV_PATH = BASE_DIR / "DR_grading.csv"
IMAGE_DIR = BASE_DIR / "DR_grading"

if not IMAGE_DIR.exists():
    IMAGE_DIR = BASE_DIR

OUTPUT_DIR = Path("outputs_ddr_effnetb0")
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "reports").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "confusion_matrices").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "splits").mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("CSV_PATH:", CSV_PATH)
print("IMAGE_DIR:", IMAGE_DIR)

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Image folder not found: {IMAGE_DIR}")



full_df = pd.read_csv(CSV_PATH)

print("CSV columns:", list(full_df.columns))
print(full_df.head())

possible_image_columns = ["id_code", "image", "image_id", "filename", "file_name", "name", "img_path", "path"]
possible_label_columns  = ["diagnosis", "level", "label", "grade", "dr_grade", "DR_grade"]

image_col = next((col for col in possible_image_columns if col in full_df.columns), None)
label_col  = next((col for col in possible_label_columns  if col in full_df.columns), None)

if image_col is None:
    raise ValueError(f"Could not find image column. Found: {list(full_df.columns)}")
if label_col is None:
    raise ValueError(f"Could not find label column. Found: {list(full_df.columns)}")

df = full_df[[image_col, label_col]].copy()
df.columns = ["image_name", "diagnosis"]
df = df.dropna(subset=["image_name", "diagnosis"]).copy()
df["diagnosis"] = df["diagnosis"].astype(int)
df = df[df["diagnosis"].between(0, 4)].copy()

print("Total samples after cleaning:", len(df))
print("Class distribution:")
print(df["diagnosis"].value_counts().sort_index())



train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["diagnosis"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["diagnosis"], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_df.to_csv(OUTPUT_DIR / "splits" / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "splits" / "val.csv",   index=False)
test_df.to_csv(OUTPUT_DIR / "splits" / "test.csv", index=False)

print("\nTrain distribution:"); print(train_df["diagnosis"].value_counts().sort_index())
print("\nValidation distribution:"); print(val_df["diagnosis"].value_counts().sort_index())
print("\nTest distribution:"); print(test_df["diagnosis"].value_counts().sort_index())



VALID_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]


def resolve_image_path(image_folder, image_name):
    image_name = str(image_name)
    candidate  = Path(image_name)

    if candidate.is_absolute() and candidate.exists():
        return candidate

    direct_path = Path(image_folder) / image_name
    if direct_path.exists():
        return direct_path

    if candidate.suffix.lower() == "":
        for ext in VALID_EXTENSIONS:
            path = Path(image_folder) / f"{image_name}{ext}"
            if path.exists():
                return path

    stem = candidate.stem
    for ext in VALID_EXTENSIONS:
        matches = list(Path(image_folder).rglob(f"{stem}{ext}"))
        if matches:
            return matches[0]

    raise FileNotFoundError(f"Image not found for CSV value: {image_name} inside {image_folder}")


def crop_black_borders(image, tolerance=7):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if image.ndim == 3 else image
    mask = gray > tolerance

    if mask.sum() == 0:
        return image

    coords = np.argwhere(mask)
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0) + 1
    return image[y_min:y_max, x_min:x_max]



class FundusDataset(Dataset):
    def __init__(self, dataframe, image_folder, transform=None, use_crop=True):
        self.dataframe    = dataframe.reset_index(drop=True)
        self.image_folder = Path(image_folder)
        self.transform    = transform
        self.use_crop     = use_crop

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.loc[idx, "image_name"]
        img_path = resolve_image_path(self.image_folder, img_name)

        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Image could not be read: {img_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.use_crop:
            image = crop_black_borders(image)

        label = int(self.dataframe.loc[idx, "diagnosis"])

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label



IMG_SIZE = 512

train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=20, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=0,
        border_mode=cv2.BORDER_CONSTANT,
        p=0.4
    ),
    A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.4),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),
    A.GaussNoise(p=0.15),
    A.CoarseDropout(
        num_holes_range=(4, 8),
        hole_height_range=(32, 64),
        hole_width_range=(32, 64),
        fill=0,
        p=0.3
    ),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_test_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])



BATCH_SIZE  = 16
NUM_WORKERS = 0

train_dataset = FundusDataset(train_df, IMAGE_DIR, transform=train_transforms,    use_crop=True)
val_dataset   = FundusDataset(val_df,   IMAGE_DIR, transform=val_test_transforms, use_crop=True)
test_dataset  = FundusDataset(test_df,  IMAGE_DIR, transform=val_test_transforms, use_crop=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())



model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=0,
    drop_rate=0.3,
    drop_path_rate=0.2
)

in_features = model.num_features

class EfficientNetWithHead(nn.Module):
    def __init__(self, backbone, in_features, num_classes=5, dropout=0.5):
        super().__init__()
        self.backbone = backbone
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(in_features),
            nn.Dropout(dropout),
            nn.Linear(in_features, 256),
            nn.SiLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone.forward_features(x)
        pooled   = self.pool(features)
        return self.head(pooled)

model = EfficientNetWithHead(model, in_features, num_classes=5, dropout=0.5)
model = model.to(device)



def freeze_backbone(m):
    for p in m.backbone.parameters():
        p.requires_grad = False

def unfreeze_backbone(m):
    for p in m.backbone.parameters():
        p.requires_grad = True

freeze_backbone(model)

print("Backbone frozen. Trainable params:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))



class_counts = train_df["diagnosis"].value_counts().sort_index()
class_counts = class_counts.reindex(range(5), fill_value=0).values

if np.any(class_counts == 0):
    raise ValueError(f"At least one class has 0 samples: {class_counts}")

total_samples  = class_counts.sum()
class_weights  = total_samples / (len(class_counts) * class_counts)

class_weights  = np.clip(class_weights, 0.3, 5.0)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class counts:", class_counts)
print("Class weights (clipped):", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)



def mixup_batch(images, labels, num_classes=5, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    batch_size = images.size(0)
    idx = torch.randperm(batch_size, device=images.device)

    mixed_images = lam * images + (1 - lam) * images[idx]

    y_a = torch.zeros(batch_size, num_classes, device=images.device).scatter_(1, labels.unsqueeze(1), 1)
    y_b = torch.zeros(batch_size, num_classes, device=images.device).scatter_(1, labels[idx].unsqueeze(1), 1)
    mixed_labels = lam * y_a + (1 - lam) * y_b

    return mixed_images, mixed_labels


def mixup_criterion(criterion_fn, outputs, mixed_labels):
    log_probs = torch.nn.functional.log_softmax(outputs, dim=1)
    loss = -(mixed_labels * log_probs).sum(dim=1).mean()
    return loss



EPOCHS_STAGE1 = 5
EPOCHS_STAGE2 = 30
LEARNING_RATE  = 1e-3
BACKBONE_LR    = 5e-5

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=2e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6, verbose=True
)



best_val_loss     = float("inf")
patience          = 8
patience_counter  = 0
best_model_path   = OUTPUT_DIR / "models" / "best_effnetb0_ddr.pth"

train_losses      = []
val_losses        = []
train_accuracies  = []
val_accuracies    = []

use_amp = torch.cuda.is_available()
scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

MIXUP_PROB    = 0.5
GRAD_CLIP_MAX = 1.0



def run_epoch(loader, is_train, use_mixup=False):
    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct    = 0
    n_samples  = 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images = images.to(device).float()
            labels = labels.to(device)

            apply_mixup = is_train and use_mixup and (random.random() < MIXUP_PROB)

            if apply_mixup:
                images, mixed_labels = mixup_batch(images, labels)

            if is_train:
                optimizer.zero_grad()

            with torch.amp.autocast("cuda", enabled=use_amp):
                outputs = model(images)
                if apply_mixup:
                    loss = mixup_criterion(criterion, outputs, mixed_labels)
                else:
                    loss = criterion(outputs, labels)

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct   += (preds == labels).sum().item()
            n_samples += labels.size(0)

    return total_loss / n_samples, correct / n_samples



print("\n" + "="*60)
print("STAGE 1 — Head warm-up (backbone frozen)")
print("="*60)

for epoch in range(EPOCHS_STAGE1):
    train_loss, train_acc = run_epoch(train_loader, is_train=True, use_mixup=False)
    val_loss,   val_acc   = run_epoch(val_loader,   is_train=False)

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"[S1] Epoch {epoch+1}/{EPOCHS_STAGE1} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print("  ✓ Best model saved.")
    else:
        patience_counter += 1



print("\n" + "="*60)
print("STAGE 2 — Full fine-tuning (backbone unfrozen)")
print("="*60)

unfreeze_backbone(model)

optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": BACKBONE_LR},
    {"params": model.head.parameters(),     "lr": LEARNING_RATE * 0.5}
], weight_decay=2e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-7, verbose=True
)

patience_counter = 0

for epoch in range(EPOCHS_STAGE2):
    train_loss, train_acc = run_epoch(train_loader, is_train=True, use_mixup=True)
    val_loss,   val_acc   = run_epoch(val_loader,   is_train=False)

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"[S2] Epoch {epoch+1}/{EPOCHS_STAGE2} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print("  ✓ Best model saved.")
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print("Early stopping triggered.")
        break



epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, train_losses, label="Train Loss")
plt.plot(epochs_range, val_losses,   label="Validation Loss")
plt.axvline(x=EPOCHS_STAGE1, color="gray", linestyle="--", label="Unfreeze point")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "reports" / "loss_curve.png", dpi=300)
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, [a * 100 for a in train_accuracies], label="Train Accuracy")
plt.plot(epochs_range, [a * 100 for a in val_accuracies],   label="Validation Accuracy")
plt.axvline(x=EPOCHS_STAGE1, color="gray", linestyle="--", label="Unfreeze point")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "reports" / "accuracy_curve.png", dpi=300)
plt.close()



class_names = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
model.to(device)
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Final Test Evaluation"):
        images  = images.to(device).float()
        outputs = model(images)
        preds   = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
report   = classification_report(
    all_labels, all_preds,
    labels=[0, 1, 2, 3, 4],
    target_names=class_names,
    digits=4,
    zero_division=0
)

print("\nFinal TEST Accuracy:", accuracy)
print(report)

with open(OUTPUT_DIR / "reports" / "classification_report_test.txt", "w", encoding="utf-8") as f:
    f.write(f"Final TEST Accuracy: {accuracy}\n\n")
    f.write(report)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3, 4])

plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix - EfficientNetB0 DDR - Test")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(np.arange(5), class_names, rotation=45, ha="right")
plt.yticks(np.arange(5), class_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices" / "confusion_matrix_test.png", dpi=300)
plt.close()

print("\nDone. Outputs saved in:", OUTPUT_DIR.resolve())

PyTorch: 2.5.1+cu121
CUDA available: True
Device: cuda
BASE_DIR: .
CSV_PATH: DR_grading.csv
IMAGE_DIR: DR_grading
CSV columns: ['id_code', 'diagnosis']
                 id_code  diagnosis
0  20170413102628830.jpg          0
1  20170413111955404.jpg          0
2  20170413112015395.jpg          0
3  20170413112017305.jpg          0
4  20170413112528859.jpg          0
Total samples after cleaning: 12522
Class distribution:
diagnosis
0    6266
1     630
2    4477
3     236
4     913
Name: count, dtype: int64

Train distribution:
diagnosis
0    4386
1     441
2    3134
3     165
4     639
Name: count, dtype: int64

Validation distribution:
diagnosis
0    940
1     94
2    671
3     36
4    137
Name: count, dtype: int64

Test distribution:
diagnosis
0    940
1     95
2    672
3     35
4    137
Name: count, dtype: int64


C:\Users\flavi\AppData\Local\Temp\ipykernel_24556\4069390310.py:224: UserWarning: Argument(s) 'value' are not valid for transform Rotate
  A.Rotate(limit=20, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5),
d:\anaconda3\envs\licenta_final\lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
d:\anaconda3\envs\licenta_final\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Backbone frozen. Trainable params: 332293
Class counts: [4386  441 3134  165  639]
Class weights (clipped): [0.3996808  3.97505669 0.55934907 5.         2.74334898]

STAGE 1 — Head warm-up (backbone frozen)


[S1] Epoch 1/5 | Train Loss: 1.8779 Acc: 35.41% | Val Loss: 1.6613 Acc: 59.32%
  ✓ Best model saved.


[S1] Epoch 2/5 | Train Loss: 1.8001 Acc: 42.60% | Val Loss: 1.6483 Acc: 53.04%
  ✓ Best model saved.


...

[S2] Epoch 18/30 | Train Loss: 1.1265 Acc: 71.08% | Val Loss: 1.5271 Acc: 86.90%
  No improvement. Patience: 7/8


[S2] Epoch 19/30 | Train Loss: 1.1615 Acc: 72.53% | Val Loss: 1.4952 Acc: 87.11%
  No improvement. Patience: 8/8
Early stopping triggered.


Final Test Evaluation: 100%|██████████| 118/118 [00:45<00:00,  2.62it/s]



Final TEST Accuracy: 0.8131985098456626
                  precision    recall  f1-score   support

           No DR     0.9238    0.8904    0.9068       940
            Mild     0.3314    0.6000    0.4270        95
        Moderate     0.8643    0.7396    0.7971       672
          Severe     0.2804    0.8571    0.4225        35
Proliferative DR     0.8992    0.7810    0.8359       137

        accuracy                         0.8132      1879
       macro avg     0.6598    0.7736    0.6779      1879
    weighted avg     0.8588    0.8132    0.8291      1879


Done. Outputs saved in: D:\licenta_Tania\outputs_ddr_effnetb0


: 